
# Sensor Data Standardization – ExacerbAlert

## 1. Introduction and Objective

### 1.1 Project Overview

ExacerbAlert is a context-aware healthcare system designed to analyze longitudinal patient data and identify changes in health indicators that may indicate potential COPD exacerbations.

This notebook focuses on the sensor data standardization stage of the project's preprocessing pipeline.

### 1.2 Dataset Information

The project uses the TOLIFE Clinical Health Indices Dataset V1.0.

**Dataset source:** https://zenodo.org/records/16642439

The dataset includes 74 participants from two countries:

- Germany: 34 participants
- Spain: 40 participants

It contains eight sensor streams:

1. BRI – Breathing Rate Index
2. EI – Environmental Index
3. MI – Mobility Index
4. PEI – Pulmonary Efficiency Index
5. PRVI – Pulse Rate Variability Index
6. SHRI – Sleeping Heart Rate Index
7. SII – Sound Intensity Index
8. SQI – Sleep Quality Index

### 1.3 Objective

The objective is to consolidate participant-level sensor CSV files into eight standardized, sensor-specific datasets.

The standardization process includes:

- Adding Participant_ID and Country metadata.
- Adding a Sensor identifier where appropriate.
- Converting timestamps into a consistent UTC datetime representation.
- Aligning column structures across participants.
- Preserving original measurements and validity flags.
- Combining participant records separately for each sensor.
- Exporting the consolidated datasets in Parquet format.

### 1.4 Data Preservation Principles

This notebook performs standardization only, not data cleaning.

The following operations are intentionally excluded:

- Removing invalid or zero-value records.
- Removing duplicate timestamps.
- Imputing missing measurements.
- Scaling or normalizing numerical values.
- Removing outliers.
- Modifying original raw CSV files.
- Combining different sensor streams based on row positions.

Missing sensor files and schema differences will be documented rather than silently corrected.

### 1.5 Expected Outputs

The standardized datasets will be saved under:

`data/interim/standardized/sensors/`

Expected deliverables:

- BRI.parquet
- EI.parquet
- MI.parquet
- PEI.parquet
- PRVI.parquet
- SHRI.parquet
- SII.parquet
- SQI.parquet
- Standardization and validation summary CSV files



## 1. Importing Libraries and Defining Paths

This section imports the required Python libraries and defines
the input and output directories for sensor data standardization.

In [ ]:

from pathlib import Path
import pandas as pd
import numpy as np

# Find the project root directory
project_path = Path.cwd()

if not (project_path / "environment.yml").exists():
    project_path = project_path.parent

# Input dataset directory
dataset_path = (
    project_path
    / "data"
    / "raw"
    / "TOLIFE_COPD_Dataset"
)

# Output directory
output_path = (
    project_path
    / "data"
    / "interim"
    / "standardized"
    / "sensors"
)

# Create output directory if it doesn't exist
output_path.mkdir(parents=True, exist_ok=True)

print("Project path:", project_path)
print("Dataset exists:", dataset_path.exists())
print("Output path:", output_path)

# Verify dataset directory
if not dataset_path.exists():
    raise FileNotFoundError(
        f"Dataset directory not found: {dataset_path}"
    )

print("Dataset directory verified successfully.")

## 2. Defining Sensor Streams and Countries

The dataset contains eight different sensor streams
collected from participants in Germany and Spain.

We define these streams and countries so that the same
loading procedure can be applied systematically.

In [ ]:

sensor_files = [
    "BRI",
    "EI",
    "MI",
    "PEI",
    "PRVI",
    "SHRI",
    "SII",
    "SQI"
]

countries = [
    "GERMANY",
    "SPAIN"
]

print("Total sensor streams:", len(sensor_files))
print("Countries:", countries)


## 3. Loading Sensor Data

This section iterates through all participant folders
in Germany and Spain.

For every participant, it checks the availability of
each sensor CSV file, loads available files, adds
Participant_ID and Country, and normalizes timestamps.

In [ ]:

# Create an empty list for every sensor
sensor_data = {
    sensor: []
    for sensor in sensor_files
}

# Track missing sensor files
missing_files = []

# Loop through countries
for country in countries:

    country_path = dataset_path / "index" / country

    if not country_path.exists():
        print("Country folder missing:", country_path)
        continue

    # Loop through participant folders
    for participant_path in sorted(country_path.iterdir()):

        if not participant_path.is_dir():
            continue

        participant_id = participant_path.name

        # Loop through all sensors
        for sensor in sensor_files:

            file_path = participant_path / f"{sensor}.csv"

            # Check whether sensor file exists
            if not file_path.exists():

                missing_files.append({
                    "Participant_ID": participant_id,
                    "Country": country,
                    "Sensor": sensor
                })

                continue

            # Load CSV file
            temp_df = pd.read_csv(file_path)

            # Add participant information
            temp_df["Participant_ID"] = participant_id
            temp_df["Country"] = country

            # Normalize timestamp to UTC
            if "Timestamp" in temp_df.columns:
                temp_df["Timestamp"] = pd.to_datetime(
                    temp_df["Timestamp"],
                    errors="coerce",
                    utc=True
                )

            # Store data according to sensor
            sensor_data[sensor].append(temp_df)

print("Sensor loading completed.")


## 4. Combining Sensor Data

All participant files belonging to the same sensor
are combined into one DataFrame.

Different sensor schemas are handled using the union
of available columns. Missing values remain missing.

In [ ]:

summary = []

for sensor, participant_frames in sensor_data.items():

    if not participant_frames:
        print("No files found for:", sensor)
        continue

    # Combine participant files
    combined_df = pd.concat(
        participant_frames,
        ignore_index=True,
        sort=False
    )

    # Save as Parquet
    output_file = output_path / f"{sensor}.parquet"

    combined_df.to_parquet(
        output_file,
        index=False
    )

    # Collect summary information
    summary.append({
        "Sensor": sensor,
        "Participants_with_files": len(participant_frames),
        "Unique_participants": combined_df["Participant_ID"].nunique(),
        "Total_rows": len(combined_df),
        "Total_columns": len(combined_df.columns)
    })

    print(sensor, "saved successfully.")

summary_df = pd.DataFrame(summary)

display(summary_df)


## 5. Saving Loading Summary

This section saves the participant counts, total rows,
and column counts for each sensor.

It also saves a separate report containing missing
sensor files.

In [ ]:

# Save sensor loading summary
summary_df.to_csv(
    output_path / "sensor_load_summary.csv",
    index=False
)

# Save missing sensor file report
missing_files_df = pd.DataFrame(missing_files)

missing_files_df.to_csv(
    output_path / "missing_sensor_files.csv",
    index=False
)

print("Sensor summary saved successfully.")
print("Missing sensor files:", len(missing_files_df))


## 6. Data Validation

This section verifies that all standardized sensor
datasets were saved correctly.

We check the number of records, participants,
and columns in every generated Parquet file.

In [ ]:

for sensor in sensor_files:

    file_path = output_path / f"{sensor}.parquet"

    if not file_path.exists():
        print(sensor, "FILE NOT FOUND")
        continue

    df = pd.read_parquet(file_path)

    print(
        sensor,
        "| Rows:", len(df),
        "| Participants:", df["Participant_ID"].nunique(),
        "| Columns:", len(df.columns)
    )


In [ ]:

for sensor in sensor_files:

    file_path = output_path / f"{sensor}.parquet"

    if not file_path.exists():
        continue

    df = pd.read_parquet(file_path)

    print("\nSensor:", sensor)

    print("Participant_ID exists:",
          "Participant_ID" in df.columns)

    print("Country exists:",
          "Country" in df.columns)

    print("data_validity exists:",
          "data_validity" in df.columns)

    print("index_validity exists:",
          "index_validity" in df.columns)

    print("Total columns:", len(df.columns))


In [ ]:
bri_df = pd.read_parquet(
    output_path / "BRI.parquet"
)

display(bri_df.head())

In [ ]:
%pip install pyarrow

In [ ]:
import pyarrow

print("PyArrow version:", pyarrow.__version__)

In [ ]:
import pyarrow
import pandas as pd

print("PyArrow:", pyarrow.__version__)
print("Pandas:", pd.__version__)

In [ ]:
import sys

print("Python executable:")
print(sys.executable)

print("\nTesting PyArrow import:")

try:
    import pyarrow
    print("PyArrow imported successfully!")
    print("Version:", pyarrow.__version__)

except Exception as e:
    print("Import failed!")
    print("Error type:", type(e).__name__)
    print("Error details:", e)

In [ ]:
import sys

print("Python executable:")
print(sys.executable)

In [ ]:
import sys

print("Python executable:")
print(sys.executable)

In [ ]:
import sys
import pandas as pd
import pyarrow

print("Python:", sys.executable)
print("Pandas:", pd.__version__)
print("PyArrow:", pyarrow.__version__)

In [ ]:
from pathlib import Path

print("Output files:")

for file in sorted(output_path.iterdir()):
    print(file.name, "-", round(file.stat().st_size / 1024, 2), "KB")

print("\nTotal files:", len(list(output_path.iterdir())))

In [ ]:
import pandas as pd

for sensor in ["BRI", "EI", "MI", "PEI", "PRVI", "SHRI", "SII", "SQI"]:
    file_path = output_path / f"{sensor}.parquet"

    if file_path.exists():
        df = pd.read_parquet(file_path)
        print(f"{sensor}: {df.shape[0]} rows, {df.shape[1]} columns")
    else:
        print(f"{sensor}: FILE MISSING")

In [ ]:
print("SENSOR LOAD SUMMARY")
display(pd.read_csv(output_path / "sensor_load_summary.csv"))

print("MISSING SENSOR FILES")
display(pd.read_csv(output_path / "missing_sensor_files.csv"))

In [ ]:
# Inspect dataset folder structure

print("Dataset folders:")

for item in dataset_path.iterdir():
    print(item.name)

In [ ]:
# Define the sensor index directory
index_path = dataset_path / "index"

# Display country folders
countries = [
    folder.name
    for folder in index_path.iterdir()
    if folder.is_dir()
]

print("Countries found:", countries)

In [ ]:
# Create participant-country mapping

participant_records = []

for country in countries:
    country_path = index_path / country

    for participant_folder in country_path.iterdir():
        if participant_folder.is_dir():
            participant_records.append({
                "Participant_ID": participant_folder.name,
                "Country": country
            })

# Convert records into a DataFrame
participants_df = pd.DataFrame(participant_records)

# Sort participant records
participants_df = participants_df.sort_values(
    by="Participant_ID"
).reset_index(drop=True)

print("Total participants:", len(participants_df))
print("\nParticipants by country:")
print(participants_df["Country"].value_counts())

print("\nFirst 10 participants:")
display(participants_df.head(10))

In [ ]:
# Save participant-country mapping

participants_file = output_path.parent / "participants_info.csv"

participants_df.to_csv(participants_file, index=False)

print("Participant information saved to:", participants_file)
print("Total records saved:", len(participants_df))

In [ ]:
# Define all sensor types

sensors = [
    "BRI", "EI", "MI", "PEI",
    "PRVI", "SHRI", "SII", "SQI"
]

print("Total sensors:", len(sensors))
print("Sensors:", sensors)

In [ ]:
# Inspect one sample CSV file per sensor

sample_participant = "GER-001"
sample_country = "GERMANY"

sample_path = index_path / sample_country / sample_participant

for sensor in sensors:
    file_path = sample_path / f"{sensor}.csv"

    print("\n" + "=" * 60)
    print("SENSOR:", sensor)

    if file_path.exists():
        sample_df = pd.read_csv(file_path)

        print("Shape:", sample_df.shape)
        print("Columns:", list(sample_df.columns))
        print("\nData types:")
        print(sample_df.dtypes)

        print("\nFirst 3 rows:")
        display(sample_df.head(3))
    else:
        print("File not found:", file_path)

## 4. Sample Sensor File Inspection

One sample CSV file from participant GER-001 was inspected for each of the eight sensor types.

### Observations

- All eight sensor files contain a Timestamp column stored as text in the original CSV files.
- PRVI contains missing measurement values, including nRMSSD.
- SQI contains additional columns named use_time and duration_movements. These columns are retained without assuming their meaning.
- Sensor files have different numbers of rows and columns.
- The index_validity and data_validity flags are present and must be preserved.
- Different sensors have different sampling frequencies.

### Standardization decisions

- Convert Timestamp into UTC-aware datetime format.
- Preserve all original measurement columns and validity flags.
- Retain missing values without imputation.
- Align different participant schemas when combining files belonging to the same sensor.
- Do not remove duplicate timestamps or invalid records.
- Keep each sensor dataset separate.

## 5. Reusable Sensor Loading and Standardization Function

A reusable Python function is created to load and standardize individual participant sensor files.

The function performs the following operations:

- Locates the required sensor CSV file.
- Loads the file into a Pandas DataFrame.
- Converts timestamps into UTC-aware datetime format.
- Adds Participant_ID, Country, and Sensor columns.
- Preserves original measurements, missing values, and validity flags.

The function does not perform data cleaning, imputation, scaling, or duplicate removal.

In [ ]:
def load_and_standardize_sensor(participant_id, country, sensor):
    """
    Load and standardize one participant's sensor CSV file.
    """

    # Locate the participant's sensor file
    file_path = (
        index_path
        / country
        / participant_id
        / f"{sensor}.csv"
    )

    # Return None if the sensor file is missing
    if not file_path.exists():
        return None

    # Load CSV file
    df = pd.read_csv(file_path)

    # Convert timestamps to UTC-aware datetime
    df["Timestamp"] = pd.to_datetime(
        df["Timestamp"],
        utc=True,
        errors="coerce"
    )

    # Add participant information
    df["Participant_ID"] = participant_id
    df["Country"] = country
    df["Sensor"] = sensor

    return df

In [ ]:
# Test the function using GER-001 and BRI

test_df = load_and_standardize_sensor(
    participant_id="GER-001",
    country="GERMANY",
    sensor="BRI"
)

print("Shape:", test_df.shape)
print("\nData types:")
print(test_df.dtypes)

print("\nFirst 5 rows:")
display(test_df.head())

### Function Testing

The reusable standardization function was tested using the BRI sensor file of participant GER-001.

The function successfully:

- Loaded all 93 original records.
- Converted timestamps into UTC-aware datetime format.
- Added Participant_ID, Country, and Sensor columns.
- Preserved all original measurement and validity columns.
- Returned a DataFrame with 10 columns.

The test confirmed that the function works as expected without modifying or removing original records.

## 6. Processing All Participants for Each Sensor

The reusable standardization function is applied to all participants and all eight sensor types.

For each sensor:

- Iterate through all 74 participants.
- Load the available participant sensor files.
- Attach participant and country information.
- Collect standardized DataFrames.
- Record missing sensor files separately.

No original records are removed or modified during this stage.

In [ ]:
# Process all participants for each sensor

sensor_data = {}
missing_files = []

for sensor in sensors:
    sensor_frames = []

    for _, participant in participants_df.iterrows():

        participant_id = participant["Participant_ID"]
        country = participant["Country"]

        df = load_and_standardize_sensor(
            participant_id,
            country,
            sensor
        )

        if df is not None:
            sensor_frames.append(df)
        else:
            missing_files.append({
                "Participant_ID": participant_id,
                "Country": country,
                "Sensor": sensor
            })

    sensor_data[sensor] = sensor_frames

    print(
        f"{sensor}: {len(sensor_frames)} files loaded, "
        f"{sum(len(df) for df in sensor_frames)} rows"
    )

# Create missing-file report
missing_sensor_df = pd.DataFrame(missing_files)

print("\nTotal available files:", sum(len(v) for v in sensor_data.values()))
print("Total missing files:", len(missing_sensor_df))

## 7. Combining Participant Data

The standardized DataFrames are combined separately for each sensor using pandas concat.

Each sensor produces one consolidated DataFrame containing records from all participants with an available file.

When participant files have different schemas, columns are aligned by name. Missing columns are represented by NaN.

No records are removed, and duplicate timestamps are retained.

In [ ]:
# Combine participant data separately for each sensor

combined_sensor_data = {}

for sensor in sensors:

    combined_df = pd.concat(
        sensor_data[sensor],
        ignore_index=True,
        sort=False
    )

    combined_sensor_data[sensor] = combined_df

    print(
        f"{sensor}: {len(combined_df)} rows, "
        f"{len(combined_df.columns)} columns, "
        f"{combined_df['Participant_ID'].nunique()} participants"
    )

print("\nTotal combined sensor records:",
      sum(len(df) for df in combined_sensor_data.values()))

## 8. Saving Standardized Sensor Datasets

The eight consolidated sensor DataFrames are saved separately in Parquet format.

Parquet is a columnar storage format that preserves data types and supports efficient data access.

All files are saved in the project's data/interim/standardized/sensors directory.

In [ ]:
# Save consolidated sensor datasets as Parquet

for sensor, df in combined_sensor_data.items():

    file_path = output_path / f"{sensor}.parquet"

    df.to_parquet(
        file_path,
        index=False,
        engine="pyarrow"
    )

    print(f"{sensor}.parquet saved successfully")

print("\nAll sensor datasets saved.")

In [ ]:
# Check generated Parquet files

parquet_files = list(output_path.glob("*.parquet"))

print("Total Parquet files:", len(parquet_files))

for file in sorted(parquet_files):
    size_kb = file.stat().st_size / 1024
    print(f"{file.name} - {size_kb:.2f} KB")

## 9. Validation of Standardized Datasets

The saved Parquet datasets are reloaded and validated to ensure that the standardization and saving process preserved the original records.

Validation includes:

- Row and column counts.
- Unique participant counts.
- Timestamp data types.
- Missing values.
- Duplicate timestamps.
- Timestamp ranges.
- Validity flag preservation.

In [ ]:
# Reload and validate saved Parquet datasets

reloaded_sensor_data = {}

for sensor in sensors:

    file_path = output_path / f"{sensor}.parquet"

    df = pd.read_parquet(file_path, engine="pyarrow")

    reloaded_sensor_data[sensor] = df

    original_rows = len(combined_sensor_data[sensor])
    saved_rows = len(df)

    print(f"\nSensor: {sensor}")
    print("Original rows:", original_rows)
    print("Saved rows:", saved_rows)
    print("Columns:", len(df.columns))
    print("Participants:", df["Participant_ID"].nunique())
    print("Row count preserved:", original_rows == saved_rows)

print("\nAll Parquet files reloaded successfully.")

In [ ]:
# Validate timestamp format and timezone

for sensor, df in reloaded_sensor_data.items():

    timestamp = df["Timestamp"]

    print(f"\nSensor: {sensor}")
    print("Timestamp dtype:", timestamp.dtype)
    print("Timezone:", timestamp.dt.tz)
    print("Missing timestamps:", timestamp.isna().sum())
    print("Earliest timestamp:", timestamp.min())
    print("Latest timestamp:", timestamp.max())

    print(
        "UTC timezone verified:",
        timestamp.dt.tz is not None
        and str(timestamp.dt.tz) == "UTC"
    )

In [ ]:
# Validate missing values and duplicate timestamps

for sensor, df in reloaded_sensor_data.items():

    print("\n" + "=" * 55)
    print("SENSOR:", sensor)

    # Count missing values
    total_missing = df.isna().sum().sum()

    print("\nTotal missing values:", total_missing)

    missing_columns = df.isna().sum()
    missing_columns = missing_columns[missing_columns > 0]

    print("Columns containing missing values:")
    if len(missing_columns) > 0:
        print(missing_columns)
    else:
        print("None")

    # Count duplicate timestamps within each participant
    duplicate_count = df.duplicated(
        subset=["Participant_ID", "Timestamp"]
    ).sum()

    print("\nDuplicate participant-timestamps:", duplicate_count)

In [ ]:
# Validate participant coverage and validity flags

for sensor, df in reloaded_sensor_data.items():

    expected_participants = (
        65 if sensor == "PRVI" else 74
    )

    actual_participants = df["Participant_ID"].nunique()

    print("\n" + "=" * 50)
    print("SENSOR:", sensor)

    print("Expected participants:", expected_participants)
    print("Actual participants:", actual_participants)

    print(
        "Participant coverage correct:",
        expected_participants == actual_participants
    )

    # Check validity flag columns
    for flag in ["index_validity", "data_validity"]:

        if flag in df.columns:
            print(f"\n{flag}:")
            print("Column preserved: True")
            print("Missing values:", df[flag].isna().sum())
            print("Observed values:", df[flag].dropna().unique())
        else:
            print(f"\n{flag}: Column not found")

### Consolidated Validation Summary

A consolidated validation summary is generated for all eight sensor datasets.

It includes:
- Total records and columns.
- Unique participant count.
- Missing-value count.
- Duplicate participant-timestamp count.
- Earliest and latest timestamps.
- UTC timestamp verification.
- Participant coverage verification.
- Validity flag preservation.

This summary provides an overview of the standardized datasets and their observed limitations.

In [ ]:
# Generate consolidated validation summary

expected_rows = {
    "BRI": 9866,
    "EI": 19732,
    "MI": 9908,
    "PEI": 9907,
    "PRVI": 2917,
    "SHRI": 9866,
    "SII": 19732,
    "SQI": 9866
}

validation_records = []

for sensor, df in reloaded_sensor_data.items():

    timestamp = df["Timestamp"]

    expected_participants = 65 if sensor == "PRVI" else 74

    timezone = timestamp.dt.tz

    validation_records.append({
        "Sensor": sensor,
        "Total_Rows": len(df),
        "Expected_Rows": expected_rows[sensor],
        "Row_Count_Match": len(df) == expected_rows[sensor],
        "Total_Columns": len(df.columns),
        "Unique_Participants": df["Participant_ID"].nunique(),
        "Expected_Participants": expected_participants,
        "Participant_Coverage": (
            df["Participant_ID"].nunique() == expected_participants
        ),
        "Missing_Values": int(df.isna().sum().sum()),
        "Duplicate_Timestamps": int(
            df.duplicated(
                subset=["Participant_ID", "Timestamp"]
            ).sum()
        ),
        "Missing_Timestamps": int(timestamp.isna().sum()),
        "Timestamp_UTC": (
            timezone is not None and str(timezone) == "UTC"
        ),
        "Validity_Flags_Preserved": all(
            flag in df.columns
            for flag in ["index_validity", "data_validity"]
        ),
        "Earliest_Timestamp": timestamp.min(),
        "Latest_Timestamp": timestamp.max()
    })

validation_summary_df = pd.DataFrame(validation_records)

display(validation_summary_df)

In [ ]:
# Save validation summary as CSV

validation_summary_path = (
    output_path.parent / "validation_summary.csv"
)

validation_summary_df.to_csv(
    validation_summary_path,
    index=False
)

print("Validation summary saved:", validation_summary_path)
print("Total sensors validated:", len(validation_summary_df))

In [ ]:
# Investigate the SQI validation summary

sqi_df = reloaded_sensor_data["SQI"]

print("Timestamp dtype:", sqi_df["Timestamp"].dtype)
print("Timestamp timezone:", sqi_df["Timestamp"].dt.tz)
print("Missing timestamps:", sqi_df["Timestamp"].isna().sum())
print("Earliest timestamp:", sqi_df["Timestamp"].min())
print("Latest timestamp:", sqi_df["Timestamp"].max())

print("\nValidity flag columns:")
print([
    flag for flag in ["index_validity", "data_validity"]
    if flag in sqi_df.columns
])

print("\nSQI validation summary row:")
display(
    validation_summary_df[
        validation_summary_df["Sensor"] == "SQI"
    ]
)

### Original CSV vs. Standardized Parquet Validation

The original CSV files are re-inspected to calculate the total number of records available for each sensor.

These counts are compared with the records stored in the standardized Parquet datasets.

The purpose is to verify that:
- All available CSV files were included.
- No records were unexpectedly lost during standardization.
- Missing sensor files are accounted for separately.

In [ ]:
# Compare original CSV records with standardized Parquet records

original_inventory = []

for sensor in sensors:

    original_rows = 0
    original_files = 0

    for _, participant in participants_df.iterrows():

        participant_id = participant["Participant_ID"]
        country = participant["Country"]

        file_path = (
            index_path
            / country
            / participant_id
            / f"{sensor}.csv"
        )

        if file_path.exists():

            df_original = pd.read_csv(file_path)

            original_rows += len(df_original)
            original_files += 1

    standardized_rows = len(reloaded_sensor_data[sensor])

    original_inventory.append({
        "Sensor": sensor,
        "Original_CSV_Files": original_files,
        "Original_CSV_Rows": original_rows,
        "Standardized_Parquet_Rows": standardized_rows,
        "Row_Count_Match": original_rows == standardized_rows
    })

original_inventory_df = pd.DataFrame(original_inventory)

display(original_inventory_df)

In [ ]:
# Merge original CSV inventory into validation summary

validation_summary_df = validation_summary_df.merge(
    original_inventory_df[
        [
            "Sensor",
            "Original_CSV_Files",
            "Original_CSV_Rows",
            "Row_Count_Match"
        ]
    ],
    on="Sensor",
    how="left",
    suffixes=("", "_Original")
)

# Rename the two row-count validation columns
validation_summary_df = validation_summary_df.rename(
    columns={
        "Row_Count_Match": "Standardized_Row_Count_Match",
        "Row_Count_Match_Original": "Original_CSV_Row_Count_Match"
    }
)

display(validation_summary_df)

### Saving the Validation Summary

The consolidated validation results are saved as a CSV file.

This report includes:
- Dataset dimensions and participant coverage.
- Timestamp and timezone validation.
- Missing values and duplicate timestamps.
- Validity flag preservation.
- Original CSV versus standardized Parquet row-count comparison.

The report provides a record of the standardization quality checks.

In [ ]:
# Save the final consolidated validation summary

validation_summary_path = (
    output_path.parent / "validation_summary.csv"
)

validation_summary_df.to_csv(
    validation_summary_path,
    index=False
)

print("Validation summary saved successfully.")
print("File location:", validation_summary_path)
print("Total sensors:", len(validation_summary_df))

print("\nOriginal vs standardized row-count checks:")
print(
    validation_summary_df[
        ["Sensor", "Original_CSV_Row_Count_Match"]
    ]
)

### Sensor Loading Summary

A loading summary is generated for all eight sensor streams.

It records:
- Number of participant files successfully loaded.
- Number of unique participants.
- Total records loaded.
- Total columns in each standardized dataset.

This report provides an overview of the sensor data available after loading and consolidation.

In [ ]:
# Generate sensor loading summary

sensor_load_records = []

for sensor, df in combined_sensor_data.items():

    sensor_load_records.append({
        "Sensor": sensor,
        "Participants_with_files": len(sensor_data[sensor]),
        "Unique_participants": df["Participant_ID"].nunique(),
        "Total_rows": len(df),
        "Total_columns": len(df.columns)
    })

sensor_load_summary_df = pd.DataFrame(sensor_load_records)

display(sensor_load_summary_df)

In [ ]:
# Save sensor loading summary as CSV

sensor_load_summary_path = (
    output_path / "sensor_load_summary.csv"
)

sensor_load_summary_df.to_csv(
    sensor_load_summary_path,
    index=False
)

print("Sensor loading summary saved successfully.")
print("File location:", sensor_load_summary_path)
print("Total sensors:", len(sensor_load_summary_df))
print(
    "Total participant sensor files:",
    sensor_load_summary_df["Participants_with_files"].sum()
)
print(
    "Total sensor records:",
    sensor_load_summary_df["Total_rows"].sum()
)

### Missing Sensor Files Report

During dataset loading, some participants were found to have missing sensor files.

A separate report is generated to record:
- Participant ID.
- Country.
- Missing sensor name.

These missing files are documented rather than treated as empty datasets or replaced with artificial values.

In [ ]:
# Generate and save missing sensor files report

missing_sensor_path = (
    output_path / "missing_sensor_files.csv"
)

missing_sensor_df.to_csv(
    missing_sensor_path,
    index=False
)

print("Missing sensor files report saved successfully.")
print("File location:", missing_sensor_path)
print("Total missing files:", len(missing_sensor_df))

display(missing_sensor_df)

### Sensor Schema Consistency Check

The column structures of the standardized sensor datasets are examined.

This check identifies:
- Total columns in each consolidated sensor dataset.
- Whether all expected validity and identification columns are present.
- Whether the original sensor-specific columns are retained.

Different sensors are expected to have different schemas. Schema differences within a sensor are documented rather than resolved by deleting potentially useful columns.

In [ ]:
# Check consolidated sensor schemas

schema_records = []

for sensor, df in combined_sensor_data.items():

    required_columns = [
        "Timestamp",
        "Participant_ID",
        "Country",
        "Sensor",
        "index_validity",
        "data_validity"
    ]

    missing_required_columns = [
        col for col in required_columns
        if col not in df.columns
    ]

    schema_records.append({
        "Sensor": sensor,
        "Total_Columns": len(df.columns),
        "Required_Columns_Present": (
            len(missing_required_columns) == 0
        ),
        "Missing_Required_Columns": ", ".join(
            missing_required_columns
        ),
        "Column_Names": ", ".join(df.columns)
    })

schema_summary_df = pd.DataFrame(schema_records)

display(
    schema_summary_df[
        [
            "Sensor",
            "Total_Columns",
            "Required_Columns_Present",
            "Missing_Required_Columns"
        ]
    ]
)

### Saving the Sensor Schema Summary

The schema verification results are saved as a CSV file.

This report records the column count and confirms whether all required identification and validity columns are present in each sensor dataset.

In [ ]:
# Save sensor schema summary

schema_summary_path = (
    output_path.parent / "schema_summary.csv"
)

schema_summary_df.to_csv(
    schema_summary_path,
    index=False
)

print("Schema summary saved successfully.")
print("File location:", schema_summary_path)
print("Total sensors checked:", len(schema_summary_df))
print(
    "All required columns present:",
    schema_summary_df["Required_Columns_Present"].all()
)

## 14. Dataset Limitations and Observations

The standardization process identified several characteristics and limitations in the TOLIFE sensor datasets.

### 14.1 Dataset Information

- The dataset contains 74 participants from Germany and Spain.
- It includes eight sensor streams: BRI, EI, MI, PEI, PRVI, SHRI, SII and SQI.
- Timestamps are documented as UTC in the dataset documentation.

### 14.2 Missing Sensor Files

- Nine PRVI files were unavailable.
- Six missing files belong to participants from Germany.
- Three missing files belong to participants from Spain.
- The remaining seven sensors have files for all 74 participants.

### 14.3 Missing Values

- Missing measurements were observed in BRI, PEI, PRVI and SQI.
- PRVI contains substantial missing measurements across several columns.
- SQI contains missing values in `use_time` and `duration_movements`.
- Missing values were preserved without imputation.

### 14.4 Duplicate Timestamps

Duplicate participant-timestamp combinations were observed in:

- EI: 70 duplicates.
- PRVI: 195 duplicates.
- SII: 70 duplicates.

These records were retained without removal.

### 14.5 Schema Differences

- PRVI contains schema variations across participant files.
- SQI contains two additional columns: `use_time` and `duration_movements`.
- The meanings of these additional SQI columns were not established from the available documentation.
- Missing columns were aligned during consolidation, with unavailable values represented as NaN.

### 14.6 Sampling and Monitoring Differences

- Sensor streams have different sampling frequencies.
- Monitoring periods and timestamp ranges vary across sensors.
- Long monitoring gaps were observed.
- Sensor datasets were kept separate rather than merged by row position.

### 14.7 Standardization Boundaries

The work in this notebook was limited to data loading, structural standardization, consolidation and validation.

The following operations were intentionally not performed:

- Missing-value imputation.
- Duplicate timestamp removal.
- Invalid or zero-value record removal.
- Outlier removal.
- Feature scaling or normalization.
- Modification of original raw CSV files.

All original measurements and available validity flags were preserved.

### 14.8 Important Note

The observations above describe the available dataset and the results of the standardization process. They do not establish the clinical meaning of missing measurements, duplicate records or undocumented columns.

These limitations should be considered during subsequent preprocessing and machine learning stages.

In [ ]:
# Create documentation directory

docs_path = project_path / "docs"

docs_path.mkdir(parents=True, exist_ok=True)

print("Documentation directory created:", docs_path)

In [ ]:
# Create standalone sensor standardization documentation

documentation = """
# ExacerbAlert – Sensor Data Standardization

## 1. Project Overview

ExacerbAlert is a collaborative healthcare machine learning project
focused on COPD-related monitoring and analysis.

This stage focuses on standardizing the TOLIFE Clinical Health
Indices Dataset V1.0, which contains sensor data collected from
74 participants across Germany and Spain.

The objective is to prepare consistent, structured sensor datasets
for subsequent preprocessing and machine learning stages.

## 2. Dataset Structure

The dataset contains eight sensor streams:

| Sensor | Description |
|---|---|
| BRI | Breathing Rate Index |
| EI | Environmental Index |
| MI | Mobility Index |
| PEI | Pulmonary Efficiency Index |
| PRVI | Pulse Rate Variability Index |
| SHRI | Sleeping Heart Rate Index |
| SII | Sound Intensity Index |
| SQI | Sleep Quality Index |

Participant distribution:
- Germany: 34 participants
- Spain: 40 participants
- Total: 74 participants

## 3. Standardization Methodology

The following operations were performed:

1. Loaded participant and country information.
2. Inspected sample CSV files from all eight sensor streams.
3. Created a reusable function to load and standardize sensor files.
4. Converted timestamps into UTC-aware datetime values.
5. Added Participant_ID, Country and Sensor columns.
6. Processed each sensor independently across available participants.
7. Combined participant records separately for each sensor.
8. Aligned differing column structures during consolidation.
9. Saved standardized datasets in Parquet format.
10. Validated the resulting datasets.

## 4. Standardization Results

A total of 583 available participant sensor files were processed,
producing 91,794 sensor records.

| Sensor | Participants | Records | Columns |
|---|---:|---:|---:|
| BRI | 74 | 9,866 | 10 |
| EI | 74 | 19,732 | 22 |
| MI | 74 | 9,908 | 14 |
| PEI | 74 | 9,907 | 14 |
| PRVI | 65 | 2,917 | 18 |
| SHRI | 74 | 9,866 | 10 |
| SII | 74 | 19,732 | 10 |
| SQI | 74 | 9,866 | 12 |
| **Total** | | **91,794** | |

## 5. Validation

The standardized datasets were checked for:

- Expected row counts and participant coverage.
- Timestamp data types and UTC timezone consistency.
- Required identification and validity columns.
- Missing-value counts.
- Duplicate participant-timestamp combinations.
- Earliest and latest timestamps.
- Original CSV versus standardized Parquet row counts.

All eight sensors passed the original CSV versus Parquet
row-count comparison.

All eight sensors contained the required identification
and validity columns.

No unexpected record loss was identified.

## 6. Dataset Limitations and Observations

### Missing sensor files

Nine PRVI files were unavailable:
- Germany: 6 participants.
- Spain: 3 participants.

The other seven sensor streams contained files for all 74 participants.

### Missing values

Missing values were observed in:
- BRI: 3
- PEI: 970
- PRVI: 8,436
- SQI: 19,738

These counts represent missing cells, not missing records.

### Duplicate timestamps

Duplicate participant-timestamp combinations:
- EI: 70
- PRVI: 195
- SII: 70

These records were retained.

### Schema differences

- PRVI showed schema variations across participant files.
- SQI contained additional columns named use_time and
  duration_movements.
- The meanings of these additional columns were not established
  from the available documentation.
- Missing columns were aligned during consolidation.

### Sampling and monitoring

- Sensor streams have different sampling frequencies.
- Monitoring periods vary across sensors.
- Long monitoring gaps were observed.

## 7. Data Preservation Rules

This stage performed structural standardization only.

The following operations were intentionally not performed:

- Missing-value imputation.
- Duplicate record removal.
- Invalid or zero-value record removal.
- Outlier removal.
- Feature scaling or normalization.
- Modification of original raw CSV files.

Original measurements and available validity flags were preserved.

## 8. Output Files

The standardized outputs are stored under:

data/interim/standardized/

Expected structure:

- participants_info.csv
- validation_summary.csv
- schema_summary.csv
- sensors/
  - BRI.parquet
  - EI.parquet
  - MI.parquet
  - PEI.parquet
  - PRVI.parquet
  - SHRI.parquet
  - SII.parquet
  - SQI.parquet
  - sensor_load_summary.csv
  - missing_sensor_files.csv

## 9. Scope of This Stage

The output of this stage is structurally standardized sensor data.

It is intended for subsequent preprocessing, data quality
handling, feature engineering and machine learning workflows.

Missing values, duplicate timestamps and other observed
limitations remain documented for later stages.
"""

documentation_path = docs_path / "sensor_standardization.md"

documentation_path.write_text(
    documentation,
    encoding="utf-8"
)

print("Documentation created successfully.")
print("File location:", documentation_path)
print("File size:", documentation_path.stat().st_size, "bytes")

In [ ]:
# Verify documentation file

print("File exists:", documentation_path.exists())

if documentation_path.exists():

    content = documentation_path.read_text(
        encoding="utf-8"
    )

    print("File size:", documentation_path.stat().st_size, "bytes")
    print("Word count:", len(content.split()))

    print("\nDocumentation headings:")

    for line in content.splitlines():
        if line.startswith("#"):
            print(line)

### Detailed Original Schema Verification

The original CSV column structures are compared across participants for each sensor.

This check identifies:
- Number of distinct column structures per sensor.
- Total unique original columns.
- Whether all original columns are represented in the consolidated dataset.

This helps verify that schema alignment did not discard sensor-specific columns.

In [ ]:
# Verify original CSV schemas against consolidated datasets

schema_audit_records = []

for sensor in sensors:

    original_schemas = set()
    original_column_union = set()
    file_count = 0

    for _, participant in participants_df.iterrows():

        participant_id = participant["Participant_ID"]
        country = participant["Country"]

        file_path = (
            index_path
            / country
            / participant_id
            / f"{sensor}.csv"
        )

        if file_path.exists():

            original_columns = pd.read_csv(
                file_path,
                nrows=0
            ).columns.tolist()

            original_schemas.add(
                tuple(sorted(original_columns))
            )

            original_column_union.update(original_columns)
            file_count += 1

    # Add metadata columns introduced during standardization
    expected_columns = original_column_union | {
        "Participant_ID",
        "Country",
        "Sensor"
    }

    consolidated_columns = set(
        combined_sensor_data[sensor].columns
    )

    schema_audit_records.append({
        "Sensor": sensor,
        "Original_Files": file_count,
        "Distinct_Original_Schemas": len(original_schemas),
        "Original_Column_Count": len(original_column_union),
        "Consolidated_Column_Count": len(consolidated_columns),
        "All_Original_Columns_Preserved": (
            expected_columns == consolidated_columns
        )
    })

schema_audit_df = pd.DataFrame(schema_audit_records)

display(schema_audit_df)

In [ ]:
# Save detailed original schema audit

schema_audit_path = (
    output_path.parent / "schema_audit_summary.csv"
)

schema_audit_df.to_csv(
    schema_audit_path,
    index=False
)

print("Detailed schema audit saved successfully.")
print("File location:", schema_audit_path)

print(
    "All original columns preserved:",
    schema_audit_df["All_Original_Columns_Preserved"].all()
)

print(
    "Sensors with schema variations:",
    (schema_audit_df["Distinct_Original_Schemas"] > 1).sum()
)

In [ ]:
# Update documentation with detailed schema audit findings

schema_documentation = """

## 10. Detailed Schema Audit

The original CSV schemas were compared across all available
participants for each sensor.

Results:

| Sensor | Original Schemas | Original Columns | Consolidated Columns |
|---|---:|---:|---:|
| BRI | 1 | 7 | 10 |
| EI | 1 | 19 | 22 |
| MI | 1 | 11 | 14 |
| PEI | 1 | 11 | 14 |
| PRVI | 2 | 15 | 18 |
| SHRI | 1 | 7 | 10 |
| SII | 1 | 7 | 10 |
| SQI | 2 | 9 | 12 |

All original columns were preserved in the consolidated datasets.

PRVI and SQI each contained two distinct original schemas.
Their column differences were retained during consolidation.

Three additional metadata columns were added to each dataset:
Participant_ID, Country and Sensor.
"""

with documentation_path.open(
    "a",
    encoding="utf-8"
) as file:
    file.write(schema_documentation)

print("Documentation updated successfully.")
print("Updated file size:", documentation_path.stat().st_size, "bytes")

In [ ]:
# Verify all expected project deliverables

standardized_path = output_path.parent
sensor_output_path = output_path

expected_files = [
    standardized_path / "participants_info.csv",
    standardized_path / "validation_summary.csv",
    standardized_path / "schema_summary.csv",
    standardized_path / "schema_audit_summary.csv",
    sensor_output_path / "sensor_load_summary.csv",
    sensor_output_path / "missing_sensor_files.csv",
    docs_path / "sensor_standardization.md"
]

# Add the eight standardized Parquet files
for sensor in sensors:
    expected_files.append(
        sensor_output_path / f"{sensor}.parquet"
    )

print("FINAL DELIVERABLE VERIFICATION\n")

for file_path in expected_files:
    status = "FOUND" if file_path.exists() else "MISSING"
    print(f"{status}: {file_path.relative_to(project_path)}")

total_found = sum(file.exists() for file in expected_files)

print("\nVerification summary:")
print("Total expected files:", len(expected_files))
print("Files found:", total_found)
print("Files missing:", len(expected_files) - total_found)
print("All deliverables present:", total_found == len(expected_files))

## 15. Independent Standardization Validation

This section independently validates the eight standardized Parquet datasets.

The validation checks:
- Whether all expected Parquet files exist and can be loaded.
- Whether row counts and participant coverage match the original dataset audit.
- Whether timestamps are UTC-aware and contain no missing values.
- Whether the original validity flags are present.
- Whether duplicate participant-timestamp combinations remain preserved.
- The distribution of missing values across individual columns.

This is an additional verification step. No records are removed, modified, imputed, or transformed during this validation.

In [ ]:

from pathlib import Path
import pandas as pd

# Locate project directory
project_path = Path.cwd()

if project_path.name == "notebooks":
    project_path = project_path.parent

sensor_path = (
    project_path / "data" / "interim" /
    "standardized" / "sensors"
)

sensors = ["BRI", "EI", "MI", "PEI", "PRVI", "SHRI", "SII", "SQI"]

expected_rows = {
    "BRI": 9866,
    "EI": 19732,
    "MI": 9908,
    "PEI": 9907,
    "PRVI": 2917,
    "SHRI": 9866,
    "SII": 19732,
    "SQI": 9866
}

expected_participants = {
    sensor: 65 if sensor == "PRVI" else 74
    for sensor in sensors
}

validation_results = []
missing_by_column = []

for sensor in sensors:

    file_path = sensor_path / f"{sensor}.parquet"

    print(f"\nChecking {sensor}...")

    if not file_path.exists():
        print("FILE MISSING")
        continue

    df = pd.read_parquet(file_path, engine="pyarrow")

    timestamp = df["Timestamp"]

    result = {
        "Sensor": sensor,
        "Rows": len(df),
        "Expected_Rows": expected_rows[sensor],
        "Row_Count_Pass": len(df) == expected_rows[sensor],
        "Participants": df["Participant_ID"].nunique(),
        "Expected_Participants": expected_participants[sensor],
        "Participant_Pass": (
            df["Participant_ID"].nunique()
            == expected_participants[sensor]
        ),
        "Timestamp_Dtype": str(timestamp.dtype),
        "UTC_Pass": (
            timestamp.dt.tz is not None
            and str(timestamp.dt.tz) == "UTC"
        ),
        "Missing_Timestamps": timestamp.isna().sum(),
        "Duplicate_Extra_Rows": df.duplicated(
            subset=["Participant_ID", "Timestamp"]
        ).sum(),
        "Missing_Cells": df.isna().sum().sum(),
        "Required_Columns_Pass": all(
            col in df.columns
            for col in [
                "Participant_ID", "Country", "Sensor",
                "index_validity", "data_validity"
            ]
        ),
        "Sensor_Label_Pass": df["Sensor"].eq(sensor).all()
    }

    validation_results.append(result)

    # Detailed missing-value report
    for column, count in df.isna().sum().items():
        if count > 0:
            missing_by_column.append({
                "Sensor": sensor,
                "Column": column,
                "Missing_Values": int(count),
                "Missing_Percentage": round(
                    count / len(df) * 100, 2
                )
            })

    print("Rows:", len(df))
    print("Participants:", df["Participant_ID"].nunique())
    print("Timestamp:", timestamp.dtype)
    print("Duplicate extra rows:", result["Duplicate_Extra_Rows"])
    print("Missing cells:", result["Missing_Cells"])

# Display results
validation_df = pd.DataFrame(validation_results)
missing_df = pd.DataFrame(missing_by_column)

print("\n========== VALIDATION SUMMARY ==========")
display(validation_df)

print("\n========== MISSING VALUES BY COLUMN ==========")
display(missing_df)

# Save separate review reports without overwriting friend's reports
review_path = project_path / "data" / "interim" / "standardized"

validation_df.to_csv(
    review_path / "independent_validation_summary.csv",
    index=False
)

missing_df.to_csv(
    review_path / "independent_missing_by_column.csv",
    index=False
)

print("\nIndependent validation reports saved.")


## 16. Independent Validation Results

### Objective
Independently verify the standardized sensor datasets against the original dataset inventory and quality audit.

### Observed Results

All eight standardized Parquet files were successfully loaded.

- Total standardized records: 91,794.
- Seven sensor streams contain 74 participants each.
- PRVI contains 65 participants, consistent with the nine missing PRVI files identified during the original audit.
- All eight streams have UTC-aware timestamps.
- No missing timestamps were detected.
- Row counts match the original dataset inventory.
- Required metadata and validity columns are present.
- Sensor labels are consistent with their respective datasets.

### Duplicate Timestamp Findings

Duplicate extra rows were found in:
- EI: 70
- PRVI: 195
- SII: 70

These findings match the original duplicate timestamp audit. Duplicate records have been preserved.

### Missing-Value Findings

Missing measurements were preserved during standardization.

The SQI columns `use_time` and `duration_movements` have 9,862 missing values each after consolidation. These columns were present in only two original participant files, so most of the missingness is caused by schema differences across participants.

PRVI also contains structural missingness because two participant files do not contain the `nRMSSD` and related range columns.

### Conclusion

The independent structural validation supports the successful completion of sensor standardization.

The standardized datasets retain their original records, missing values, duplicate timestamps, measurement columns and validity flags.

No cleaning, imputation, duplicate removal or outlier treatment was performed during this validation.

**Status:** Core sensor standardization approved. Detailed validity-flag value verification remains pending.

In [ ]:

flag_results = []

for sensor in sensors:
    df = pd.read_parquet(sensor_path / f"{sensor}.parquet")

    for column in ["index_validity", "data_validity"]:
        values = df[column]

        flag_results.append({
            "Sensor": sensor,
            "Flag": column,
            "Zero_Count": (values == 0).sum(),
            "One_Count": (values == 1).sum(),
            "Missing_Count": values.isna().sum(),
            "Unexpected_Values": sorted(
                values.dropna().unique().tolist()
                and set(values.dropna().unique().tolist()) - {0, 1}
            )
        })

flag_validation_df = pd.DataFrame(flag_results)

display(flag_validation_df)
